In [16]:
from google.colab import files

uploaded = files.upload()

for filename in uploaded.keys():
    print(f'User uploaded file "{filename}" with length {len(uploaded[filename])} bytes')


In [15]:
import sys
import pandas as pd # Import pandas for Excel reading


# ══════════════════════════════════════════════════════════════════
#  CONFIGURACION
# ══════════════════════════════════════════════════════════════════
ArchivoEntrada = "/content/LabAnalisis.xlsx" # Path to the uploaded Excel file
TamañoBloque = 2                          # tamano de bloque para el ordenamiento interno
NumeroCintas = 2                          # cintas usadas en la mezcla equilibrada multiple
MetodoInterno = "insercion"               # "burbuja" | "seleccion" | "insercion"
MostrarDetallePasadas = False             # True = imprime cada pasada/nivel de la mezcla
AnchoReporte = 66                         # ancho minimo (en caracteres) de los encabezados


# ══════════════════════════════════════════════════════════════════
#  LECTURA DE ENTRADA
# ══════════════════════════════════════════════════════════════════
def leerCadenasDesdeTxt(rutaArchivo):
    if rutaArchivo.lower().endswith('.xlsx'):
        try:
            df = pd.read_excel(rutaArchivo)
            cadenas = df.iloc[:, 0].astype(str).tolist()
        except FileNotFoundError:
            print(f"ERROR: no se encontro el archivo '{rutaArchivo}'.")
            sys.exit(1)
        except Exception as e:
            print(f"ERROR al leer el archivo Excel '{rutaArchivo}': {e}")
            sys.exit(1)
    else:
        try:
            with open(rutaArchivo, "r", encoding="utf-8") as f:
                cadenas = [linea.strip() for linea in f if linea.strip()]
        except FileNotFoundError:
            print(f"ERROR: no se encontro el archivo '{rutaArchivo}'.")
            sys.exit(1)

    if not cadenas:
        print(f"ERROR: el archivo '{rutaArchivo}' esta vacio o no contiene cadenas validas.")
        sys.exit(1)

    return cadenas


# ══════════════════════════════════════════════════════════════════
#  CRITERIO DE ORDEN
# ══════════════════════════════════════════════════════════════════
def clave(c):
    return ord(c.lower())


# ══════════════════════════════════════════════════════════════════
#  FASE 1 - ORDENAMIENTO INTERNO
# ══════════════════════════════════════════════════════════════════
def burbuja(bloque):
    b = bloque[:]
    n = len(b)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if clave(b[j]) > clave(b[j + 1]):
                b[j], b[j + 1] = b[j + 1], b[j]
    return b


def seleccion(bloque):
    b = bloque[:]
    n = len(b)
    for i in range(n - 1):
        idxMin = i
        for j in range(i + 1, n):
            if clave(b[j]) < clave(b[idxMin]):
                idxMin = j
        b[i], b[idxMin] = b[idxMin], b[i]
    return b


def insercion(bloque):
    b = bloque[:]
    for i in range(1, len(b)):
        actual = b[i]
        j = i - 1
        while j >= 0 and clave(b[j]) > clave(actual):
            b[j + 1] = b[j]
            j -= 1
        b[j + 1] = actual
    return b


METODOS_INTERNOS = {
    "burbuja": burbuja,
    "seleccion": seleccion,
    "insercion": insercion,
}


def generarBloques(cadena, tamBloque, metodo):
    ordenar = METODOS_INTERNOS[metodo]
    bloquesOriginales = [list(cadena[i:i + tamBloque])
                          for i in range(0, len(cadena), tamBloque)]
    bloquesOrdenados = [ordenar(b) for b in bloquesOriginales]
    return bloquesOriginales, bloquesOrdenados


# ══════════════════════════════════════════════════════════════════
#  UTILIDADES DE FUSION
# ══════════════════════════════════════════════════════════════════
def merge(runA, runB):
    i, j = 0, 0
    resultado = []
    while i < len(runA) and j < len(runB):
        if clave(runA[i]) <= clave(runB[j]):
            resultado.append(runA[i]); i += 1
        else:
            resultado.append(runB[j]); j += 1
    resultado.extend(runA[i:])
    resultado.extend(runB[j:])
    return resultado


def fmt(corridas):
    return [("DUMMY" if r is None else "".join(r)) for r in corridas]


# ══════════════════════════════════════════════════════════════════
#  FASE 2A - MEZCLA EQUILIBRADA MULTIPLE
# ══════════════════════════════════════════════════════════════════
def mezclaEquilibrada(corridas, numCintas, verbose=False):
    corridas = corridas[:]
    pasada = 1
    while len(corridas) > 1:
        cintas = [[] for _ in range(numCintas)]
        for idx, corrida in enumerate(corridas):
            cintas[idx % numCintas].append(corrida)

        if verbose:
            print(f"\n      Pasada {pasada} - distribucion:")
            for k, cinta in enumerate(cintas, start=1):
                print(f"        Cinta {k}: {fmt(cinta)}")

        nuevasCorridas = []
        maxLen = max(len(c) for c in cintas)
        for pos in range(maxLen):
            grupo = [cinta[pos] for cinta in cintas if pos < len(cinta)]
            fusionada = grupo[0]
            for otra in grupo[1:]:
                fusionada = merge(fusionada, otra)
            nuevasCorridas.append(fusionada)

        if verbose:
            print(f"        Resultado: {fmt(nuevasCorridas)}")

        corridas = nuevasCorridas
        pasada += 1

    return corridas[0]


# ══════════════════════════════════════════════════════════════════
#  FASE 2B - METODO POLIFASICO
# ══════════════════════════════════════════════════════════════════
def fibonacciHasta(minimo):
    fibs = [1, 1]
    while fibs[-1] < minimo:
        fibs.append(fibs[-1] + fibs[-2])
    return fibs


def distribucionPolifasica(n):
    fibs = fibonacciHasta(n)
    totalPerfecto = fibs[-1]
    dummies = totalPerfecto - n
    a = fibs[-2]
    b = fibs[-3] if len(fibs) >= 3 else 0
    return a, b, dummies


def mezclaPolifasica(corridas, verbose=False):
    n = len(corridas)
    a, b, dummies = distribucionPolifasica(n)

    t1 = corridas[:a]
    t2 = corridas[a:a + b] + [None] * dummies
    t3 = []

    if verbose:
        print(f"\n      Distribucion inicial (Fibonacci): "
              f"T1={a}, T2={b}+{dummies} dummy(s), T3=0")

    nivel = 1
    while len(t1) > 0 and len(t2) > 0:
        if verbose:
            print(f"\n      Nivel {nivel}:")
            print(f"        T1: {fmt(t1)}")
            print(f"        T2: {fmt(t2)}")

        numMerges = min(len(t1), len(t2))
        for i in range(numMerges):
            r1, r2 = t1[i], t2[i]
            if r1 is None:
                fusion = r2
            elif r2 is None:
                fusion = r1
            else:
                fusion = merge(r1, r2)
            t3.append(fusion)

        if verbose:
            print(f"        T3 (salida): {fmt(t3)}")

        sobrante = t1[numMerges:] + t2[numMerges:]
        t1, t2, t3 = sobrante, t3, []
        nivel += 1

    resultado = [r for r in (t1 + t2) if r is not None]
    return resultado[0] if resultado else None


# ══════════════════════════════════════════════════════════════════
#  FASE 3 - ELIMINACION DE DUPLICADOS
# ══════════════════════════════════════════════════════════════════
def eliminarDuplicados(secuenciaOrdenada):
    resultado = []
    eliminados = []
    for elem in secuenciaOrdenada:
        if resultado and clave(resultado[-1]) == clave(elem):
            eliminados.append(elem)
        else:
            resultado.append(elem)
    return resultado, eliminados


# ══════════════════════════════════════════════════════════════════
#  UTILIDADES DE PRESENTACION
# ══════════════════════════════════════════════════════════════════
def encabezado(texto):
    ancho = max(AnchoReporte, len(texto) + 4)
    print("\n╔" + "═" * (ancho - 2) + "╗")
    print("║" + texto.center(ancho - 2) + "║")
    print("╚" + "═" * (ancho - 2) + "╝")


def subtitulo(texto):
    relleno = max(2, AnchoReporte - len(texto) - 6)
    print(f"\n  ── {texto} " + "─" * relleno)


def imprimirTabla(encabezados, filas, sangria="  "):
    numCols = len(encabezados)
    anchos = [len(str(encabezados[c])) for c in range(numCols)]
    for fila in filas:
        for c in range(numCols):
            anchos[c] = max(anchos[c], len(str(fila[c])))

    def borde(izq, mid, der):
        return sangria + izq + mid.join("─" * (a + 2) for a in anchos) + der

    def filaTexto(valores):
        celdas = [str(v).ljust(anchos[c]) for c, v in enumerate(valores)]
        return sangria + "│ " + " │ ".join(celdas) + " │"

    print(borde("┌", "┬", "┐"))
    print(filaTexto(encabezados))
    print(borde("├", "┼", "┤"))
    for fila in filas:
        print(filaTexto(fila))
    print(borde("└", "┴", "┘"))


# ══════════════════════════════════════════════════════════════════
#  PROCESAMIENTO DE UNA CADENA
# ══════════════════════════════════════════════════════════════════
def procesarCadena(cadena, indice, total):
    encabezado(f'CADENA {indice} de {total}  →  "{cadena}"')

    bloquesOriginales, bloquesOrdenados = generarBloques(cadena, TamañoBloque, MetodoInterno)

    subtitulo(f"Fase 1 · Bloques (tamaño {TamañoBloque}, método: {MetodoInterno})")
    filasBloques = [(f"B{i}", ",".join(o), ",".join(s)) for i, (o, s) in
                     enumerate(zip(bloquesOriginales, bloquesOrdenados), start=1)]
    imprimirTabla(["Bloque", "Original", "Ordenado"], filasBloques)

    subtitulo("Fase 2 · Ordenamiento externo")
    resEquilibrada = mezclaEquilibrada(bloquesOrdenados, NumeroCintas, verbose=MostrarDetallePasadas)
    resPolifasico = mezclaPolifasica(bloquesOrdenados, verbose=MostrarDetallePasadas)
    coinciden = resEquilibrada == resPolifasico

    etiquetasFase2 = [
        f"Mezcla equilibrada ({NumeroCintas} cintas)",
        "Método polifásico (3 cintas)",
        "¿Coinciden ambos métodos?",
    ]
    anchoEtq2 = max(len(e) for e in etiquetasFase2)
    print(f"    • {etiquetasFase2[0].ljust(anchoEtq2)} : {','.join(resEquilibrada)}")
    print(f"    • {etiquetasFase2[1].ljust(anchoEtq2)} : {','.join(resPolifasico)}")
    print(f"    • {etiquetasFase2[2].ljust(anchoEtq2)} : {'Sí ✓' if coinciden else 'No ✗'}")

    subtitulo("Fase 3 · Duplicados")
    resultadoSinDup, eliminados = eliminarDuplicados(resEquilibrada)
    etiquetasFase3 = ["Duplicados encontrados", "Resultado final (sin duplicados)"]
    anchoEtq3 = max(len(e) for e in etiquetasFase3)
    if eliminados:
        detalle = f"{len(eliminados)}  →  {', '.join(eliminados)}"
    else:
        detalle = "0 (ninguno)"
    print(f"    • {etiquetasFase3[0].ljust(anchoEtq3)} : {detalle}")
    print(f"    • {etiquetasFase3[1].ljust(anchoEtq3)} : {','.join(resultadoSinDup)}")

    return {
        "cadena": cadena,
        "coinciden": "Sí" if coinciden else "No",
        "numDuplicados": len(eliminados),
        "eliminados": ",".join(eliminados) if eliminados else "-",
        "resultadoFinal": ",".join(resultadoSinDup),
    }


def imprimirResumenGeneral(resumenes):
    encabezado("RESUMEN GENERAL")
    filas = [(i, r["cadena"], r["coinciden"], r["numDuplicados"],
              r["eliminados"], r["resultadoFinal"])
             for i, r in enumerate(resumenes, start=1)]
    imprimirTabla(["#", "Cadena", "OK", "Dup.", "Eliminados", "Resultado final"], filas)


# ══════════════════════════════════════════════════════════════════
#  PROGRAMA PRINCIPAL
# ══════════════════════════════════════════════════════════════════
if __name__ == "__main__":
    rutaArchivo = ArchivoEntrada
    cadenas = leerCadenasDesdeTxt(rutaArchivo)

    print(f"Archivo de entrada: '{rutaArchivo}'  ({len(cadenas)} cadena(s) encontrada(s))")

    resumenes = [procesarCadena(cadena, i, len(cadenas))
                 for i, cadena in enumerate(cadenas, start=1)]

    if len(resumenes) > 1:
        imprimirResumenGeneral(resumenes)


Archivo de entrada: '/content/LabAnalisis.xlsx'  (700 cadena(s) encontrada(s))

╔════════════════════════════════════════════════════════════════╗
║                 CADENA 1 de 700  →  "DMB2427D"                 ║
╚════════════════════════════════════════════════════════════════╝

  ── Fase 1 · Bloques (tamaño 2, método: insercion) ──────────────
  ┌────────┬──────────┬──────────┐
  │ Bloque │ Original │ Ordenado │
  ├────────┼──────────┼──────────┤
  │ B1     │ D,M      │ D,M      │
  │ B2     │ B,2      │ 2,B      │
  │ B3     │ 4,2      │ 2,4      │
  │ B4     │ 7,D      │ 7,D      │
  └────────┴──────────┴──────────┘

  ── Fase 2 · Ordenamiento externo ───────────────────────────────
    • Mezcla equilibrada (2 cintas) : 2,2,4,7,B,D,D,M
    • Método polifásico (3 cintas)  : 2,2,4,7,B,D,D,M
    • ¿Coinciden ambos métodos?     : Sí ✓

  ── Fase 3 · Duplicados ─────────────────────────────────────────
    • Duplicados encontrados           : 2  →  2, D
    • Resultado final (sin dupli